<a href="https://colab.research.google.com/github/albijanashala/ML1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
%pip -q install duckdb

import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FEB = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("Connected")

Connected


In [2]:
DECISION_DATE = "2026-02-28"

features = con.sql(f"""
    WITH feb AS (
        SELECT client_hash_id,
               content_hash_id,
               SUM(gsc_impressions) AS imp_feb,
               SUM(gsc_clicks) AS clicks_feb,
               SUM(gsc_sum_position) AS sum_pos_feb,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_visible_feb,
               MIN(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS best_pos_feb,
               SUM(CASE WHEN report_date <= DATE '2026-02-14' THEN gsc_impressions ELSE 0 END) AS imp_h1,
               SUM(CASE WHEN report_date >  DATE '2026-02-14' THEN gsc_impressions ELSE 0 END) AS imp_h2,
               MAX(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS has_ga4
        FROM {FEB}
        GROUP BY 1, 2
    ),
    mar AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_mar
        FROM {MAR}
        GROUP BY 1, 2
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.imp_feb,
        f.clicks_feb,
        f.days_visible_feb,
        ROUND(f.clicks_feb::FLOAT / NULLIF(f.imp_feb, 0), 5) AS ctr_feb,
        ROUND(f.imp_feb::FLOAT / NULLIF(f.days_visible_feb, 0), 2) AS imp_per_active_day,
        ROUND(f.sum_pos_feb::FLOAT / NULLIF(f.imp_feb, 0), 2) AS pos_feb,
        COALESCE(f.best_pos_feb, 100) AS best_pos_feb,
        ROUND(f.imp_h2::FLOAT / NULLIF(f.imp_h1, 0), 3) AS trend_within_feb,
        f.has_ga4,
        COALESCE(d.word_count, 0) AS word_count,
        CASE WHEN d.word_count IS NULL THEN 1 ELSE 0 END AS word_count_missing,
        COALESCE(d.search_volume, 0) AS search_volume,
        COALESCE(d.competition, 0) AS competition,
        COALESCE(d.backlinks, 0) AS backlinks,
        DATE_DIFF('day', d.content_created_date, DATE '{DECISION_DATE}') AS content_age_days,
        COALESCE(d.content_type, 'unknown') AS content_type,
        COALESCE(d.competition_level, 'unknown') AS competition_level,
        COALESCE(d.main_intent, 'unknown') AS main_intent,
        COALESCE(m.imp_mar, 0) AS imp_mar
    FROM feb f
    LEFT JOIN mar m USING (client_hash_id, content_hash_id)
    LEFT JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
    WHERE f.imp_feb >= 50
    ORDER BY f.client_hash_id, f.content_hash_id
""").df()

print(f"Rows: {len(features):,}")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 93,654


,client_hash_id,content_hash_id,imp_feb,clicks_feb,days_visible_feb,ctr_feb,imp_per_active_day,pos_feb,best_pos_feb,trend_within_feb,...,word_count,word_count_missing,search_volume,competition,backlinks,content_age_days,content_type,competition_level,main_intent,imp_mar
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,246.0,1.0,28,0.00407,8.79,17.24,8.50,0.662,...,3168,0,0,0.00,0,144,keyword article,LOW,informational,331.0
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,137.0,0.0,27,0.00000,5.07,9.66,7.75,1.076,...,4135,0,20,0.03,9,144,keyword article,LOW,commercial,33.0
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,121.0,0.0,28,0.00000,4.32,9.12,5.75,0.833,...,3211,0,0,0.00,0,144,keyword article,LOW,informational,145.0
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,174.0,0.0,12,0.00000,14.50,13.38,9.00,NaN,...,3465,0,0,0.00,0,144,keyword article,LOW,informational,461.0
4,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,164.0,0.0,28,0.00000,5.86,11.48,7.00,0.451,...,3149,0,0,0.00,0,144,keyword article,LOW,informational,232.0


In [3]:
df = features.copy()

# February has 28 days, March has 31 — compare daily rates, not raw totals
FEB_DAYS, MAR_DAYS = 28, 31
imp_rate_feb = df["imp_feb"] / FEB_DAYS
imp_rate_mar = df["imp_mar"] / MAR_DAYS
df["is_declining"] = (imp_rate_mar < 0.8 * imp_rate_feb).astype(int)

# Explicit fills, so no rows are dropped
df["no_h1_impressions"] = df["trend_within_feb"].isna().astype(int)
df["trend_within_feb"] = df["trend_within_feb"].fillna(1.0)
df["pos_feb"] = df["pos_feb"].fillna(100)
df["content_age_days"] = df["content_age_days"].fillna(-1)

print(f"Rows: {len(df):,}")
print(f"Positive rate: {df['is_declining'].mean():.1%} ({df['is_declining'].sum():,} declining)")

Rows: 93,654
Positive rate: 27.6% (25,887 declining)


In [4]:
# Population gate first: pages with enough impressions to judge (session slide 10)
eligible = df[df["imp_feb"] >= 100].copy()

# Compare like with like: CTR is only meaningful against peers at a similar position
eligible["pos_bucket"] = pd.cut(
    eligible["pos_feb"],
    bins=[0, 3, 10, 20, 50, 1000],
    labels=["1-3", "4-10 (page 1)", "11-20 (page 2)", "21-50", "50+"],
)

check_ctr = eligible.groupby("pos_bucket", observed=True).agg(
    n=("ctr_feb", "size"),
    median_ctr_pct=("ctr_feb", lambda s: round(s.median() * 100, 3)),
    declining_pct=("is_declining", lambda s: round(s.mean() * 100, 1)),
)
print("Signal 1 — CTR by position bucket (belief behind the CTR-fix flag)")
print(check_ctr.to_string())

Signal 1 — CTR by position bucket (belief behind the CTR-fix flag)
                    n  median_ctr_pct  declining_pct
pos_bucket                                          
1-3             12092           0.189           28.8
4-10 (page 1)   40880           0.186           26.1
11-20 (page 2)  15902           0.073           27.8
21-50           10237           0.000           23.9
50+              1210           0.000           36.8


**Signal 1 verdict: MIXED, and the position column itself failed inspection.**

The bucket table shows CTR falling as position worsens — median 0.189% at positions
1-3 down to 0.000% beyond position 20 — so the belief behind the CTR-fix flag is
directionally present. But two problems stop it carrying a rule.

First, positions 1-3 and 4-10 are almost identical (0.189% vs 0.186%, n=12,092 and
n=40,880), so page-one positions do not separate. Second, and more seriously, the
position column is not trustworthy. Inspecting a single page day by day shows
`gsc_avg_position` values below 1 — for example 3,519 impressions with
`gsc_sum_position` 524, giving an average position of 0.149. Position 1 is the best
rank Google awards, so a value below 1 is not a real position. `gsc_sum_position`
does not reconcile with `gsc_impressions` on high-traffic days, and the warehouse's
own `gsc_avg_position` column carries the same impossible values.

Consequence for my rule: I drop position entirely and build on signals I have
verified — impressions, clicks, CTR, days visible, and within-February momentum.
The data-quality finding is documented in section 4.

In [5]:
age_bucket = pd.cut(
    eligible["content_age_days"],
    bins=[-2, 90, 180, 365, 100000],
    labels=["0-90d", "91-180d", "181-365d", "365d+"],
)

check_age = eligible.groupby(age_bucket, observed=True).agg(
    n=("is_declining", "size"),
    declining_pct=("is_declining", lambda s: round(s.mean() * 100, 1)),
    median_imp=("imp_feb", "median"),
)
print("Signal 2 — decline rate by content age (belief behind the refresh flags)")
print(check_age.to_string())

Signal 2 — decline rate by content age (belief behind the refresh flags)
                      n  declining_pct  median_imp
content_age_days                                  
0-90d             20899           19.6       557.0
91-180d           16987           30.7       785.0
181-365d          35840           29.3       787.0
365d+              6596           25.3       637.0


**Signal 2 verdict: MIXED.**

Decline rate rises from 19.6% (0-90 days, n=20,899) to 30.7% (91-180 days,
n=16,987), then falls back to 25.3% for pages over a year old (n=6,596). The refresh
belief — older content declines more — holds in the first half and reverses at the
top end.

A plausible reading is survivorship: a page still earning 100+ impressions after a
year has already proven durable, while fragile old pages fell below my eligibility
gate long ago. I cannot test that here, so it stays a hypothesis.

Consequence for my rule: age cannot carry a refresh decision alone. It needs a
partner signal, which is what the session recommended for a MIXED verdict.

**The rule, read as one sentence.**

AMONG pages with at least 500 February impressions and at least 7 active days —
the population, large enough that a few clicks cannot flip the verdict —
LOOK AT click-through rate against the median CTR of pages in the same impression
band — the evidence — IF CTR is below half that band median — the condition —
THEN send it to snippet review with reason code CTR_BELOW_VOLUME_PEERS — the action.

Pages that pass the population gate but fail that condition fall to a second branch:
if within-February momentum is below 0.8, flag them REFRESH_MOMENTUM_LOSS. Everything
else is MONITOR.

**Why impression bands instead of position bands.** The session compared CTR against
position peers, and that is the better comparison in principle. I cannot make it here
because the position column fails inspection (section 1, signal 1). Impression volume
is the closest verified proxy: pages earning similar traffic face broadly similar
competition, so comparing their CTR is a fairer test than comparing raw CTR across
the whole portfolio.

**Why these thresholds.** Half the band median is a starting point, not a truth. The
500-impression floor and 7-day minimum come from the session's 38-impression warning:
a page with too little traffic should never reach the threshold test at all.

**Reason codes:** `CTR_BELOW_VOLUME_PEERS`, `REFRESH_MOMENTUM_LOSS`, `MONITOR`.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
import json
from pathlib import Path

OUT_DIR = Path("work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Population gate: enough traffic and enough active days to judge
MIN_IMPRESSIONS = 500
MIN_ACTIVE_DAYS = 7
queue = df[(df["imp_feb"] >= MIN_IMPRESSIONS) & (df["days_visible_feb"] >= MIN_ACTIVE_DAYS)].copy()

# Compare like with like: CTR against peers earning similar traffic
queue["imp_band"] = pd.qcut(queue["imp_feb"], q=5,
                            labels=["p0-20", "p20-40", "p40-60", "p60-80", "p80-100"])
band_median_ctr = queue.groupby("imp_band", observed=True)["ctr_feb"].median()
queue["peer_ctr"] = queue["imp_band"].map(band_median_ctr).astype(float)
queue["ctr_ratio"] = queue["ctr_feb"] / queue["peer_ctr"].replace(0, float("nan"))

CTR_THRESHOLD = 0.5
MOMENTUM_THRESHOLD = 0.8

is_ctr_case = (queue["ctr_ratio"] < CTR_THRESHOLD) & queue["trend_within_feb"].between(0.5, 3.0)
is_refresh_case = (~is_ctr_case) & (queue["trend_within_feb"] < MOMENTUM_THRESHOLD)

queue["reason_code"] = "MONITOR"
queue.loc[is_refresh_case, "reason_code"] = "REFRESH_MOMENTUM_LOSS"
queue.loc[is_ctr_case, "reason_code"] = "CTR_BELOW_VOLUME_PEERS"

queue["action"] = queue["reason_code"].map({
    "CTR_BELOW_VOLUME_PEERS": "snippet_review",
    "REFRESH_MOMENTUM_LOSS": "content_refresh",
    "MONITOR": "monitor",
})

# Score: what is at stake if we fix it
queue["missed_clicks"] = (queue["peer_ctr"] - queue["ctr_feb"]).clip(lower=0) * queue["imp_feb"]
queue["momentum_penalty"] = (1 - queue["trend_within_feb"]).clip(lower=0, upper=1)
queue["score"] = (queue["missed_clicks"] * (1 + queue["momentum_penalty"])).round(2)
queue.loc[queue["reason_code"] == "MONITOR", "score"] = 0.0

queue = queue.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

export_cols = ["rank", "score", "action", "reason_code", "client_hash_id", "content_hash_id",
               "imp_feb", "clicks_feb", "ctr_feb", "peer_ctr", "ctr_ratio", "imp_band",
               "days_visible_feb", "trend_within_feb", "content_age_days", "word_count",
               "missed_clicks"]
queue[export_cols].to_csv(OUT_DIR / "baseline_action_score.csv", index=False)

print(f"Eligible population : {len(queue):,} of {len(df):,} pages")
print("\nAction distribution:")
print(queue["action"].value_counts().to_string())
print(f"\nWritten to {OUT_DIR / 'baseline_action_score.csv'}")
queue[["rank", "score", "action", "reason_code", "imp_feb", "clicks_feb",
       "ctr_feb", "peer_ctr", "trend_within_feb"]].head(10)

Eligible population : 46,637 of 93,654 pages

Action distribution:
action
monitor            29228
snippet_review     12366
content_refresh     5043

Written to work/outputs/baseline_action_score.csv


,rank,score,action,reason_code,imp_feb,clicks_feb,ctr_feb,peer_ctr,trend_within_feb
0,1,215.65,snippet_review,CTR_BELOW_VOLUME_PEERS,81768.0,34.0,0.00042,0.0024,0.668
1,2,186.95,snippet_review,CTR_BELOW_VOLUME_PEERS,66357.0,22.0,0.00033,0.0024,0.639
2,3,178.10,snippet_review,CTR_BELOW_VOLUME_PEERS,116407.0,101.0,0.00087,0.0024,2.216
3,4,169.99,snippet_review,CTR_BELOW_VOLUME_PEERS,72478.0,39.0,0.00054,0.0024,0.739
4,5,166.62,snippet_review,CTR_BELOW_VOLUME_PEERS,59434.0,13.0,0.00022,0.0024,0.714
5,6,148.91,snippet_review,CTR_BELOW_VOLUME_PEERS,83657.0,52.0,0.00062,0.0024,1.236
6,7,144.00,content_refresh,REFRESH_MOMENTUM_LOSS,36484.0,6.0,0.00016,0.0024,0.238
7,8,140.59,content_refresh,REFRESH_MOMENTUM_LOSS,37771.0,19.0,0.00050,0.0024,0.041
8,9,135.92,snippet_review,CTR_BELOW_VOLUME_PEERS,79986.0,74.0,0.00093,0.0024,0.844
9,10,131.40,snippet_review,CTR_BELOW_VOLUME_PEERS,51205.0,35.0,0.00068,0.0024,0.508


In [7]:
metrics = {
    "run_date": pd.Timestamp.now().strftime("%Y-%m-%d"),
    "feature_window": "2026-02",
    "label_window": "2026-03",
    "total_pages": int(len(df)),
    "eligible_population": int(len(queue)),
    "thresholds": {
        "min_impressions": MIN_IMPRESSIONS,
        "min_active_days": MIN_ACTIVE_DAYS,
        "ctr_ratio": CTR_THRESHOLD,
        "momentum": MOMENTUM_THRESHOLD,
        "ctr_branch_trend_guard": [0.5, 3.0],
    },
    "action_counts": queue["action"].value_counts().to_dict(),
    "peer_ctr_by_band": {str(k): float(v) for k, v in band_median_ctr.items()},
    "signal_verdicts": {"ctr_vs_position": "MIXED", "content_age": "MIXED"},
    "excluded_columns": ["gsc_avg_position", "gsc_sum_position"],
}

with open(OUT_DIR / "w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))

{
  "run_date": "2026-08-09",
  "feature_window": "2026-02",
  "label_window": "2026-03",
  "total_pages": 93654,
  "eligible_population": 46637,
  "thresholds": {
    "min_impressions": 500,
    "min_active_days": 7,
    "ctr_ratio": 0.5,
    "momentum": 0.8,
    "ctr_branch_trend_guard": [
      0.5,
      3.0
    ]
  },
  "action_counts": {
    "monitor": 29228,
    "snippet_review": 12366,
    "content_refresh": 5043
  },
  "peer_ctr_by_band": {
    "p0-20": 0.0015200000489130616,
    "p20-40": 0.0016899999463930726,
    "p40-60": 0.0019600000232458115,
    "p60-80": 0.002529999939724803,
    "p80-100": 0.002400000113993883
  },
  "signal_verdicts": {
    "ctr_vs_position": "MIXED",
    "content_age": "MIXED"
  },
  "excluded_columns": [
    "gsc_avg_position",
    "gsc_sum_position"
  ]
}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
print(queue[["rank", "score", "action", "reason_code", "imp_feb", "clicks_feb",
             "ctr_feb", "ctr_ratio"]].head(10).to_string())
print()
print(queue[["rank", "days_visible_feb", "trend_within_feb",
             "content_age_days", "word_count"]].head(10).to_string())

   rank   score           action             reason_code   imp_feb  clicks_feb  ctr_feb  ctr_ratio
0     1  215.65   snippet_review  CTR_BELOW_VOLUME_PEERS   81768.0        34.0  0.00042   0.175000
1     2  186.95   snippet_review  CTR_BELOW_VOLUME_PEERS   66357.0        22.0  0.00033   0.137500
2     3  178.10   snippet_review  CTR_BELOW_VOLUME_PEERS  116407.0       101.0  0.00087   0.362500
3     4  169.99   snippet_review  CTR_BELOW_VOLUME_PEERS   72478.0        39.0  0.00054   0.225000
4     5  166.62   snippet_review  CTR_BELOW_VOLUME_PEERS   59434.0        13.0  0.00022   0.091667
5     6  148.91   snippet_review  CTR_BELOW_VOLUME_PEERS   83657.0        52.0  0.00062   0.258333
6     7  144.00  content_refresh   REFRESH_MOMENTUM_LOSS   36484.0         6.0  0.00016   0.066667
7     8  140.59  content_refresh   REFRESH_MOMENTUM_LOSS   37771.0        19.0  0.00050   0.208333
8     9  135.92   snippet_review  CTR_BELOW_VOLUME_PEERS   79986.0        74.0  0.00093   0.387500
9    10  1

| # | Action | Why it's here | What would make it wrong |
|---|---|---|---|
| 1 | snippet_review | 81,768 impressions, 34 clicks — 17.5% of band median CTR, momentum 0.668 | If the page ranks for broad queries with weak intent, the low CTR reflects query mix rather than a poor snippet, and a rewrite would not move it |
| 2 | snippet_review | 66,357 impressions, 22 clicks — 13.8% of peers | The page is 47 days old, so February is close to its first full month indexed. Early CTR is often low while rankings settle, and a rewrite would be measured against a moving baseline |
| 3 | snippet_review | 116,407 impressions, 101 clicks — 36% of peers, highest volume in the ten | 101 clicks is the most of any row here. A page this broad may be performing adequately on its target term while the impression count is inflated by loosely related queries. `word_count` is also missing, so a content problem cannot be ruled out |
| 4 | snippet_review | 72,478 impressions, 39 clicks — 22.5% of peers, momentum 0.739 | Momentum is already falling. If the decline is a ranking loss rather than a click-capture problem, the snippet is not the cause and fixing it treats a symptom |
| 5 | snippet_review | 59,434 impressions, 13 clicks — 9.2% of peers, weakest ratio in the ten | `word_count` is missing, so I cannot tell whether this is a thin page. A ratio this low is as consistent with content that does not match intent as with a weak snippet |
| 6 | snippet_review | 83,657 impressions, 52 clicks — 25.8% of peers | Momentum is 1.236 — the page is growing. Flagging a page that is already recovering risks spending review time on something fixing itself, and any post-fix measurement would credit the rewrite for a trend that predates it |
| 7 | content_refresh | Momentum 0.238 — lost 76% of impressions across February | A drop this steep in two weeks is more consistent with a ranking event, deindexing, or a URL change than with content staleness. A refresh would not fix any of those. `word_count` is also missing |
| 8 | content_refresh | Momentum 0.041 — lost 96% of impressions across February | Near-total collapse. Almost certainly structural rather than editorial — the page may have been redirected, deindexed, or cannibalised by another page. Refreshing content would address none of these |
| 9 | snippet_review | 79,986 impressions, 74 clicks — 38.8% of peers, mildest case here | At 0.39 of the band median this is the closest to the 0.5 threshold. A small change to the threshold or to the band boundaries would drop it out of the queue entirely, so its inclusion is less robust than the rows above it |
| 10 | snippet_review | 51,205 impressions, 35 clicks — 28.3% of peers, momentum 0.508 | Momentum is halving while CTR is low. If both stem from a single cause — losing position on a main query — then the snippet is the wrong lever and the CTR branch has misrouted this page |

**What the review found.** Six of the ten have a specific reason the recommendation
could be wrong, and two patterns recur. Three rows have missing `word_count`, so a
content problem cannot be ruled out for them. And the two refresh cases both lost
more than 75% of their impressions inside a single month, which looks structural
rather than editorial — my rule cannot tell a stale page from a deindexed one.

Rows 2 and 6 are the ones I would remove first: one is too new to judge, the other
is already growing.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [9]:
FEATURE_WINDOW = "2026-02"
LABEL_WINDOW = "2026-03"

rule_inputs = {
    "imp_feb": FEATURE_WINDOW,
    "clicks_feb": FEATURE_WINDOW,
    "ctr_feb": FEATURE_WINDOW,
    "days_visible_feb": FEATURE_WINDOW,
    "trend_within_feb": FEATURE_WINDOW,
    "peer_ctr": f"{FEATURE_WINDOW} (median of eligible pages)",
    "imp_band": FEATURE_WINDOW,
}

march_sourced = [c for c, src in rule_inputs.items() if LABEL_WINDOW in src]
label_derived = [c for c in rule_inputs if c in ("imp_mar", "is_declining")]
product_flags = [c for c in queue.columns
                 if any(t in c.lower() for t in ("health_score", "priority", "action_type", "flag"))]

print(f"Inputs to the rule        : {len(rule_inputs)}")
print(f"Sourced from March        : {len(march_sourced)}")
print(f"Label-derived             : {len(label_derived)}")
print(f"Product flags in the frame: {len(product_flags)}")

assert not march_sourced, "No rule input may come from the label window"
assert not label_derived, "No rule input may be derived from the label"
assert not product_flags, "No FlyRank product flag may enter the rule"

# The label exists in the frame, but must not appear in the exported queue
leaked_in_csv = [c for c in export_cols if c in ("imp_mar", "is_declining")]
print(f"Label columns in the CSV  : {len(leaked_in_csv)}")
assert not leaked_in_csv, "The exported queue must not carry the label"

print("\nLeakage check passed: the rule uses February inputs only.")

Inputs to the rule        : 7
Sourced from March        : 0
Label-derived             : 0
Product flags in the frame: 0
Label columns in the CSV  : 0

Leakage check passed: the rule uses February inputs only.


**Which picks look wrong, and why.**

Rows 2 and 6 are the weakest in the top ten. Row 2 is 47 days old, so February is
close to its first full month indexed and low early CTR is expected while rankings
settle. Row 6 has momentum 1.236 — it is growing, and flagging a recovering page
spends review time on something already fixing itself.

Rows 7 and 8 are weak in a different way. Both lost more than 75% of their February
impressions (momentum 0.238 and 0.041), and a collapse that steep looks structural —
deindexing, a redirect, cannibalisation — rather than editorial. My rule labels them
`content_refresh` because momentum is the only decline signal it has. A refresh would
not fix any of the likelier causes.

**Two data limits behind the weak picks.**

Three of the top ten have `word_count = 0`, which is my fill for missing rather than
a measured zero. For those rows I cannot separate a snippet problem from a thin-content
problem, so the action label is less certain than it looks.

The peer CTR benchmark is not monotonic across impression bands: median CTR rises from
0.152% (p0-20) to 0.253% (p60-80) and then falls to 0.240% in the top band. The largest
pages earn slightly worse click capture than the band below them, which weakens the
assumption that impression volume is a clean proxy for comparable competition.
Directional only — I have not tested why.

**The position column was excluded on evidence, not preference.** Inspecting one page
day by day showed `gsc_avg_position` values below 1 — 3,519 impressions with
`gsc_sum_position` 524 gives an average position of 0.149. Position 1 is the best rank
Google awards, so a value below 1 is not a real position. `gsc_sum_position` does not
reconcile with `gsc_impressions` on high-traffic days. I dropped position from the rule
and rebuilt the peer comparison on impression volume instead, which is why my reason
code is `CTR_BELOW_VOLUME_PEERS` rather than the session's position-based version.

**Leakage.** The rule reads February impressions, clicks, CTR, active days, and
within-February momentum, plus a peer median computed from the same window. March
appears in the notebook only as `imp_mar`, used to build `is_declining` for the signal
checks; it is not an input to the score and is not exported in the queue. No FlyRank
product flag (`health_score`, `priority_score`, `action_type`) exists in the release,
so none could enter. The cell above asserts all of this rather than claiming it.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.